# Gobernanza, Seguridad y Cumplimiento (Senior)

Objetivos:
- Definir un marco práctico de gobernanza alineado a negocio.
- Implementar controles de seguridad (RBAC, ABAC, clasificación, cifrado).
- Integrar cumplimiento (GDPR, CCPA) en el ciclo de datos.
- Diseñar auditoría y trazabilidad (lineage + logging).

## 1. Gobernanza: Componentes Clave

Pilares:
1. Políticas (definiciones y alcance).
2. Roles y Responsabilidades (stewards, custodians, owners).
3. Procesos (ingestión, calidad, acceso, cambio de esquema).
4. Métricas (calidad, frescura, cumplimiento de SLAs).
5. Herramientas (catálogo, lineage, quality suite).

Modelo RACI breve ejemplo: 
- Steward: Responsable de calidad y significado.
- Owner: Aprueba cambios y acceso.
- Custodian (TI): Implementa controles técnicos.

## 1.1. Roles y Responsabilidades Detalladas

**Roles de Gobernanza** (basado en DAMA-DMBOK):

| Rol | Responsabilidades | Ejemplo |
|-----|-------------------|---------|
| **Data Owner** | Define reglas de negocio, aprueba accesos, responsable del valor del dato | Gerente Comercial (owner de `clientes`) |
| **Data Steward** | Calidad, significado, metadatos; puente entre negocio y TI | Analista de Calidad |
| **Data Custodian** | Implementación técnica, backups, seguridad física/lógica | Equipo Infraestructura |
| **Data Governance Lead** | Estrategia, políticas, comité de gobierno, métricas | Arquitecto de Datos Senior |
| **Consumers** | Usuarios finales, analistas, científicos de datos | Equipo BI, Marketing |

**Matriz RACI (ejemplo para cambio de esquema):**

| Actividad | Owner | Steward | Custodian | Governance Lead | Consumers |
|-----------|-------|---------|-----------|-----------------|-----------|
| Proponer cambio | I | R | C | C | I |
| Evaluar impacto | C | A | R | C | I |
| Aprobar cambio | A | C | I | C | I |
| Implementar | I | I | R | I | I |
| Comunicar | I | R | C | A | I |

R=Responsable, A=Aprueba, C=Consultado, I=Informado


## 1.2. Políticas de Datos

**Política = regla/directriz que establece cómo se gestionan los datos.**

### Ejemplos de Políticas

1. **Política de Clasificación:**
   - Todo dataset debe clasificarse: Público, Interno, Confidencial, Regulado.
   - Clasificación por defecto = Interno hasta revisión.

2. **Política de Acceso:**
   - Acceso basado en "need-to-know" (mínimo privilegio).
   - Revisión trimestral de permisos.
   - Aprobación del Data Owner + Seguridad para datos Regulados.

3. **Política de Retención:**
   - Datos transaccionales: 7 años (compliance fiscal).
   - Logs: 90 días en caliente, 1 año en archivo.
   - PII: borrado dentro de 30 días de solicitud (GDPR).

4. **Política de Calidad:**
   - Silver y Gold requieren validación (quality gate) antes de publicación.
   - SLO de frescura: Gold diario listo antes de 07:00.
   - Alertas automáticas si calidad < umbral (ej. nulls > 5%).

### Enforcement

- Automatización: quality gates en CI/CD, RBAC en catálogo/warehouse.
- Monitoreo: dashboards de cumplimiento (% datasets clasificados, auditorías pendientes).
- Excepciones: proceso formal de aprobación y logging.


## 1.3. Procedimientos (Workflows)

**Procedimiento = secuencia de pasos para aplicar una política.**

### Ejemplo 1: Solicitud de Acceso a Dataset

1. Usuario completa formulario (dataset, justificación, alcance).
2. Sistema valida clasificación → si Regulado, requiere aprobación adicional.
3. Data Owner revisa y aprueba/rechaza.
4. Si aprobado, Custodian provisiona acceso (RBAC).
5. Auditoría registra acceso otorgado (quién, cuándo, por qué).
6. Revisión periódica (90 días) para renovar o revocar.

### Ejemplo 2: Cambio de Esquema (Breaking Change)

1. Propuesta con justificación e impacto estimado.
2. Steward evalúa impacto en downstream (lineage).
3. Notificación a consumidores afectados (mín. 2 semanas).
4. Governance Lead aprueba (committee si es crítico).
5. Implementación en dev → QA → staging → prod.
6. Post-mortem: documentar incidentes y ajustar proceso.

### Ejemplo 3: Proceso de Calidad (Quality Gate)

1. Dataset llega a zona Silver.
2. Ejecución automática de suite de calidad (nulls, duplicados, esquema).
3. Si falla test crítico → alerta + bloqueo de publicación a Gold.
4. Steward investiga y corrige origen (Bronze o ingesta).
5. Re-ejecución tras corrección.
6. Aprobación manual (si aplica) y promoción a Gold.


## 1.4. Herramientas de Gobernanza

| Categoría | Herramientas | Uso |
|-----------|--------------|-----|
| **Catálogo** | Atlan, Collibra, Alation, DataHub, Apache Atlas | Inventario, búsqueda, documentación, lineage |
| **Calidad** | Great Expectations, Soda, dbt tests, Monte Carlo | Validación automática, alertas, monitoreo drift |
| **Lineage** | OpenLineage, Marquez, dbt lineage, Snowflake lineage | Rastreo origen-destino, análisis de impacto |
| **Acceso/Seguridad** | Ranger, Privacera, Immuta, BigQuery IAM | RBAC/ABAC, masking dinámico, audit logs |
| **Workflow** | Airflow, Prefect, Dagster | Orquestación pipelines + hooks de calidad |

**Integración típica:**
- Catálogo ← metadatos técnicos (auto-discovery) + metadatos de negocio (manual).
- Quality suite → genera métricas → catálogo las expone.
- Lineage recolectado vía OpenLineage → Marquez → catálogo visualiza.


In [ ]:
# Simulación de métricas de gobierno
import pandas as pd

# Inventario de datasets (simulado)
datasets = pd.DataFrame({
    'dataset': ['bronze.ventas_raw', 'silver.ventas_limpias', 'gold.kpi_ventas', 'bronze.clientes_raw', 'silver.clientes'],
    'clasificacion': ['Interno', 'Interno', 'Confidencial', 'Regulado', 'Regulado'],
    'owner': ['Comercial', 'Comercial', 'Comercial', 'Comercial', 'Comercial'],
    'steward_asignado': [True, True, True, False, True],
    'calidad_validada': [False, True, True, False, True],
    'acceso_revisado_90d': [False, True, True, False, False]
})

# Métricas de cumplimiento
pct_steward = datasets['steward_asignado'].mean() * 100
pct_calidad = datasets['calidad_validada'].mean() * 100
pct_acceso_rev = datasets['acceso_revisado_90d'].mean() * 100

print('=== Dashboard de Gobierno ===')
print(f'Datasets con Steward asignado: {pct_steward:.0f}%')
print(f'Datasets con calidad validada: {pct_calidad:.0f}%')
print(f'Datasets con acceso revisado (90d): {pct_acceso_rev:.0f}%')
print('\nDatasets sin Steward:')
print(datasets[~datasets['steward_asignado']]['dataset'].tolist())


## 2. Clasificación y Protección de Datos

Clasificaciones típicas: Público, Interno, Confidencial, Regulados (PII/PCI/PHI).

Controles:
- Data Masking (enmascaramiento dinámico).
- Tokenización (reemplazo reversible controlado).
- Pseudonimización (irrevocable o difícil de revertir).
- Cifrado at-rest (KMS) y in-transit (TLS).

Ejemplo de mapeo PII: nombre, email, teléfono, dirección.

## 2.1. Seguridad desde el Diseño (Security by Design)

**Principio:** Integrar seguridad en todas las fases del ciclo de vida de datos, no como capa posterior.

### Fases y Controles

| Fase | Controles de Seguridad |
|------|------------------------|
| **1. Ingesta** | Validación de fuente (TLS), autenticación (API keys/OAuth), rate limiting, sanitización de inputs |
| **2. Almacenamiento** | Encriptación at-rest (KMS), segregación por clasificación (s3://prod-regulado/ vs s3://prod-interno/), acceso mínimo privilegio |
| **3. Procesamiento** | Transformaciones seguras (evitar inyección), logs de auditoría, cifrado de datos temporales |
| **4. Exposición** | RBAC/ABAC, masking dinámico, rate limiting APIs, TLS in-transit, validación de salida |
| **5. Retención/Borrado** | Políticas de retención automáticas, derecho al olvido (soft/hard delete), backups cifrados aislados |

### Arquitectura en Capas

```
[Bronze: TLS ingesta + cifrado + RBAC restrictivo]
   ↓ ETL seguro (logs, validación)
[Silver: Clasificación + masking + lineage]
   ↓ Transformación con calidad
[Gold: ABAC contextual + DDM + auditoría completa]
   ↓
[Consumo: API con autenticación + throttling + logs]
```

### Segregación de Funciones

- **Separar ambientes:** dev, staging, prod (credenciales independientes).
- **Separar roles:** quien desarrolla pipelines ≠ quien aprueba accesos.
- **Principio de "break-glass":** Acceso de emergencia con auditoría reforzada.


In [ ]:
# Ejemplo simple de enmascaramiento de PII en pandas
import pandas as pd
df = pd.DataFrame({
  'customer_id':[1,2,3],
  'nombre':['Ana García','Luis Pérez','María López'],
  'email':['ana@example.com','luis@example.com','maria@example.com'],
  'telefono':['555-1111','555-2222','555-3333'],
  'importe_total':[1200.50,300.00,450.75]
})
def mask_email(e):
    user, domain = e.split('@'); return user[0]+'***@'+domain
def mask_nombre(n):
    partes = n.split(' '); return partes[0][0]+'*** '+partes[-1][0]+'***'
masked = df.copy()
masked['email'] = masked['email'].apply(mask_email)
masked['nombre'] = masked['nombre'].apply(mask_nombre)
masked['telefono'] = '***-****'
print('Original:
', df)
print('
Enmascarado:
', masked)

## 3. Modelos de Acceso: RBAC vs ABAC

RBAC (Role-Based Access Control): Permisos por rol.
ABAC (Attribute-Based Access Control): Evaluación de atributos (usuario, recurso, contexto).

Ejemplo ABAC: Permitir acceso si (pais=="CO" AND clearance=="ALTO").

Recomendación: Híbrido → RBAC para simplicidad + ABAC para excepciones/reglas finas.

## 4. Cumplimiento (GDPR / CCPA)

Elementos clave:
- Minimización de datos.
- Derecho al olvido / borrado selectivo.
- Consentimiento explícito y revocación.
- Registro de procesamientos (auditoría).

Flujo derecho al olvido: solicitud → verificación → localización de registros → borrado → evidencia registro.

## 4.1. Cumplimiento Normativo Detallado

### GDPR (General Data Protection Regulation - Unión Europea)

**Alcance:** Cualquier organización que procese datos de residentes UE.

**Principios clave:**
- **Legalidad:** Base legal para procesamiento (consentimiento, contrato, interés legítimo, obligación legal).
- **Transparencia:** Informar al titular sobre procesamiento (privacy notice).
- **Minimización:** Solo datos necesarios para finalidad declarada.
- **Exactitud:** Mantener datos actualizados.
- **Limitación almacenamiento:** Retener solo tiempo necesario.
- **Integridad y confidencialidad:** Seguridad técnica y organizacional.

**Derechos del titular:**
- Acceso, rectificación, supresión ("derecho al olvido").
- Portabilidad (formato estructurado, machine-readable).
- Oposición al procesamiento.
- No ser sujeto a decisiones automatizadas (ej. scoring sin revisión humana).

**Obligaciones técnicas:**
- **Privacy by Design:** Integrar privacidad desde arquitectura.
- **Data Protection Impact Assessment (DPIA):** Evaluar riesgos antes de procesamiento alto riesgo.
- **Notificación de brechas:** 72 horas a autoridad de control si hay riesgo para derechos.
- **DPO (Data Protection Officer):** Obligatorio si procesamiento masivo de datos sensibles.

**Multas:** Hasta 20M EUR o 4% ingresos anuales globales (lo mayor).


### CCPA (California Consumer Privacy Act - EE.UU.)

**Alcance:** Empresas que operan en California con ingresos >$25M, datos de >50K consumidores o >50% ingresos de venta de datos.

**Derechos del consumidor:**
- **Conocer:** Qué datos se recopilan y cómo se usan.
- **Borrar:** Solicitar eliminación de datos personales.
- **Opt-out:** Rechazar venta de datos personales a terceros.
- **No discriminación:** No penalizar por ejercer derechos.

**Obligaciones:**
- Aviso de privacidad claro.
- Mecanismo de opt-out ("Do Not Sell My Personal Information").
- Notificación de brechas (sin plazo específico como GDPR).

**Diferencias vs GDPR:**
- CCPA aplica a "venta" de datos (no solo procesamiento).
- Multas menores: hasta $7,500 por violación intencional.
- No requiere DPO ni DPIA obligatorio.


### Colombia: Ley 1581 de 2012 y Decreto 1377 de 2013

**Alcance:** Protección de datos personales en Colombia (Habeas Data).

**Principios:**
- **Legalidad:** Procesamiento con finalidad legítima.
- **Finalidad:** Informar al titular sobre uso de datos.
- **Libertad:** Consentimiento previo, expreso e informado.
- **Veracidad:** Datos completos, exactos, actualizados.
- **Transparencia:** Titular debe poder acceder a sus datos.
- **Acceso y circulación restringida:** Solo para finalidad autorizada.
- **Seguridad:** Medidas técnicas y administrativas.
- **Confidencialidad:** Obligación de todos los actores.

**Derechos del titular:**
- Conocer, actualizar y rectificar datos.
- Solicitar prueba de autorización otorgada.
- Revocar autorización (salvo obligación legal).
- Acceder gratuitamente a sus datos (mínimo 1 vez al mes).

**Obligaciones del Responsable:**
- Registrar bases de datos ante SIC (Superintendencia de Industria y Comercio).
- Designar persona responsable de atención de consultas/reclamos.
- Política de tratamiento de datos publicada.
- Autorización previa del titular (excepciones: datos públicos, orden judicial, emergencia médica).

**Datos sensibles (categoría especial):**
- Origen racial/étnico, orientación política/religiosa/filosófica, afiliación sindical, salud, vida sexual, biométricos.
- Requieren autorización **explícita** y medidas reforzadas de seguridad.

**Sanciones SIC:**
- Multas hasta 2,000 salarios mínimos mensuales (~$2M USD).
- Suspensión de actividades relacionadas con datos.

**Comparación con GDPR:**
- Similar en derechos y principios.
- Colombia: énfasis en consentimiento explícito (GDPR tiene bases legales alternativas).
- GDPR: multas más altas, alcance extraterritorial más claro.


## 4.2. Monitoreo y Reporte de Cumplimiento

### Componentes de Programa de Compliance

1. **Inventario de Datos Personales:**
   - Qué datos se recopilan, dónde se almacenan, quién accede, cuánto tiempo se retienen.
   - Mapeo de flujos de datos (data mapping / ROPA - Record of Processing Activities).

2. **Evaluaciones de Impacto (DPIA):**
   - Obligatorio para procesamiento alto riesgo (GDPR Art. 35).
   - Evaluar: necesidad, proporcionalidad, riesgos, medidas de mitigación.

3. **Políticas y Procedimientos:**
   - Política de privacidad pública.
   - Procedimiento de atención de derechos (acceso, rectificación, supresión).
   - Procedimiento de notificación de brechas.

4. **Capacitación:**
   - Personal técnico: seguridad, anonimización, logging.
   - Personal de negocio: principios de privacidad, manejo de solicitudes.

5. **Auditorías y Monitoreo Continuo:**
   - Revisiones trimestrales/anuales de cumplimiento.
   - Dashboards de métricas: % solicitudes atendidas en SLA, brechas detectadas, datasets sin clasificar.

6. **Gestión de Incidentes:**
   - Detección → Contención → Evaluación de impacto → Notificación (si aplica) → Remediación → Post-mortem.


In [ ]:
# Dashboard de Compliance (simulación)
import pandas as pd
from datetime import datetime, timedelta

# Métricas de cumplimiento
solicitudes_dsar = pd.DataFrame({
    'solicitud_id': [1, 2, 3, 4, 5],
    'tipo': ['Acceso', 'Supresión', 'Rectificación', 'Portabilidad', 'Acceso'],
    'fecha_recepcion': pd.to_datetime(['2025-11-01', '2025-11-02', '2025-11-03', '2025-11-04', '2025-11-05']),
    'fecha_respuesta': pd.to_datetime(['2025-11-05', '2025-11-10', None, '2025-11-12', '2025-11-08']),
    'sla_dias': [7, 7, 7, 7, 7],
    'estado': ['Completada', 'Completada', 'En proceso', 'Completada', 'Completada']
})

# Calcular días de respuesta
solicitudes_dsar['dias_respuesta'] = (
    solicitudes_dsar['fecha_respuesta'] - solicitudes_dsar['fecha_recepcion']
).dt.days

# Cumplimiento SLA (GDPR: 1 mes, simplificado a 7 días en ejemplo)
solicitudes_dsar['cumple_sla'] = solicitudes_dsar['dias_respuesta'] <= solicitudes_dsar['sla_dias']

print('=== Dashboard de Cumplimiento ===\n')
print('Solicitudes DSAR (Data Subject Access Requests):')
print(solicitudes_dsar[['solicitud_id', 'tipo', 'dias_respuesta', 'cumple_sla', 'estado']])

completadas = solicitudes_dsar[solicitudes_dsar['estado'] == 'Completada']
cumplimiento_sla = completadas['cumple_sla'].mean() * 100
print(f'\n% Cumplimiento SLA: {cumplimiento_sla:.0f}%')

# Otras métricas
metricas_compliance = {
    'Datasets clasificados': '95%',
    'Datasets con DPIA completado (alto riesgo)': '100% (3/3)',
    'Brechas notificadas (último trimestre)': 0,
    'Auditorías pendientes': 2,
    'Personal capacitado en GDPR': '87%'
}
print('\nMétricas adicionales:')
for k, v in metricas_compliance.items():
    print(f'  {k}: {v}')


In [ ]:
# Simulación 'derecho al olvido' simple
solicitudes_borrado = [2]  # customer_id a eliminar
post_borrado = df[~df['customer_id'].isin(solicitudes_borrado)]
print('Antes:
', df)
print('
Después de borrado (right to be forgotten):
', post_borrado)

## 5. Auditoría y Lineage

Auditoría: Registro de quién accede, qué modifica, cuándo.
Lineage: Visualización de origen→transformaciones→destino.

Prácticas:
- IDs de ejecución (run_id).
- Trazabilidad de columnas (column-level lineage).
- Registro de versiones de esquema.

Herramientas: OpenLineage, Marquez, Atlas.

## 6. Métricas de Gobernanza

Ejemplos:
- % datasets con steward asignado.
- % cobertura de catálogo.
- Tiempo promedio de aprobación de acceso.
- % cumplimiento de políticas (sin PII sin cifrar).

## 7. Ejercicio

Define para tu organización:
1. Tabla RACI para 3 dominios de datos.
2. Política de clasificación (4 niveles) y controles asociados.
3. Flujo operativo para derecho al olvido (actores y pasos).
4. Métricas iniciales de gobernanza (mínimo 4).

In [ ]:
plantilla = '''
# Solución (borrador)
RACI: [dominio -> roles]
Clasificación: [niveles + controles]
Borrado: [flujo]
Métricas: [lista]
'''
print(plantilla)

## 8. Resumen

- Gobernanza = personas + políticas + procesos + herramientas.
- Seguridad: combinar RBAC y ABAC + protección PII.
- Cumplimiento requiere operacionalizar derechos y evidencias.
- Auditoría y lineage habilitan confianza y trazabilidad.

---
Referencias:
- DAMA-DMBOK 2.0
- ISO/IEC 27001
- GDPR / CCPA textos oficiales
- NIST Privacy Framework

## 8. Ejercicio Integrador: Gobierno en Retail Omnicanal

**Contexto:** RetailCorp consolida datos de ERP, e-commerce, CRM y POS en un Lakehouse (Databricks). Problemas identificados: datasets sin clasificar, accesos descontrolados, calidad inconsistente, sin lineage.

**Dominio:** Ventas

**Datasets:**
1. `bronze.pos_transacciones_raw` (5M filas, con email_cliente opcional)
2. `bronze.ecommerce_pedidos_raw` (2M filas, con nombre/email/dirección)
3. `silver.ventas_unificadas` (6.8M filas, cliente_id hasheado)
4. `gold.kpi_ventas_diarias` (365 filas, sin PII)
5. `gold.clientes_perfil_360` (1.2M filas, con email/teléfono/historial)

**Ver caso completo:** `../04-recursos/casos-uso/caso_integrador_gobierno_retail.md`


### Tareas del Ejercicio

**2.1. Asignar Roles:**
- Para cada dataset, asigna Data Owner, Steward, Custodian.
- Considera: Gerente Comercial, Analista Calidad, Equipo Infra, Arquitecto Datos, Equipo BI, Marketing.

**2.2. Clasificar Datasets:**
- Usa categorías: Público, Interno, Confidencial, Regulado.
- Justifica cada clasificación (presencia de PII, sensibilidad).

**2.3. Matriz RACI (cambio de esquema):**
- Escenario: agregar columna `metodo_pago` a `bronze.ecommerce_pedidos_raw`.
- Actividades: Proponer, Evaluar impacto, Notificar, Aprobar, Implementar, Documentar.
- Asigna R/A/C/I a cada rol involucrado.

**2.4. Construir Lineage:**
- Dibuja grafo: fuentes → Bronze → Silver → Gold → consumidores.
- Indica transformaciones en cada paso.

**2.5. Política de Calidad:**
- Define 3 reglas para `silver.ventas_unificadas` (ej. no duplicados por pedido_id).
- SLO: nulls_cliente_id < 1%.
- Procedimiento: ¿qué pasa si falla quality gate?

**2.6. Workflow de Acceso:**
- Marketing solicita acceso a `gold.clientes_perfil_360` (Regulado).
- Diseña flujo de 6-7 pasos (solicitud → aprobaciones → provisión → auditoría → revisión).


In [ ]:
# Plantilla para solución (ejemplo parcial)
import pandas as pd

# 2.1 Roles (tabla ejemplo)
roles = pd.DataFrame({
    'dataset': ['bronze.pos_transacciones_raw', 'bronze.ecommerce_pedidos_raw', 
                'silver.ventas_unificadas', 'gold.kpi_ventas_diarias', 'gold.clientes_perfil_360'],
    'owner': ['Gerente Comercial', 'Gerente Comercial', 'Gerente Comercial', 'Equipo BI', 'Equipo Marketing'],
    'steward': ['Analista Calidad', 'Analista Calidad', 'Analista Calidad', 'Analista Calidad', 'Analista Calidad'],
    'custodian': ['Equipo Infra']*5
})

# 2.2 Clasificación
clasificacion = pd.DataFrame({
    'dataset': roles['dataset'],
    'clasificacion': ['Regulado', 'Regulado', 'Confidencial', 'Interno', 'Regulado'],
    'justificacion': [
        'Contiene email_cliente (PII)',
        'Contiene nombre, email, dirección (PII)',
        'cliente_id hasheado (pseudo-PII), datos estratégicos',
        'Agregados sin PII, uso interno',
        'Email, teléfono, historial (PII completo)'
    ]
})

print('=== Roles ===')
print(roles.to_string(index=False))
print('\n=== Clasificación ===')
print(clasificacion.to_string(index=False))

# Completa el resto de tareas (RACI, lineage, reglas de calidad, workflow)


## Actualización 2025: Gobernanza, Seguridad y Cumplimiento Adaptativo

### Evolución Clave 2025
La gobernanza pasa de catálogos estáticos y controles manuales a un modelo dinámico basado en políticas como código, linaje activo y clasificación automática asistida por ML, integrando riesgo, costo y ética en la toma de decisiones de datos y IA.

### Pilares Modernos
| Pilar | Objetivo | Herramientas / Estándares |
|-------|----------|---------------------------|
| Políticas como Código | Versionar y testear reglas | OPA (Rego), Apache Ranger, AWS Lake Formation, Immuta, Privacera |
| Linaje + Impacto | Trazabilidad para análisis de cambio | OpenLineage + DataHub / Marquez |
| Clasificación & PII | Etiquetado automático y verificación | Azure Purview, GCP Data Catalog, AWS Glue + NLP clasificadores |
| Acceso Dinámico | ABAC, RLS, Column/Row Masking | Ranger Tag Sync, Dynamic Data Masking (SQL), Attribute Stores |
| Calidad & Confianza | Medir salud preventiva | Great Expectations, Soda, dbt tests |
| Ética / IA Responsable | Sesgo, explicabilidad, retención | Model Cards, Fairlearn, Evidently AI |
| FinOps & Compliance | Balance costo/riesgo | Cost dashboards + policy drift alerts |
| Resiliencia Legal | Respuesta a nuevas normativas | GDPR, CCPA, LGPD, AI Act (UE) |

### Métricas de Gobernanza (KPIs)
| Métrica | Descripción | Meta 2025 |
|---------|-------------|-----------|
| coverage_owner_pct | % activos con owner asignado | > 95% |
| policy_test_pass_rate | % políticas que pasan tests automáticos | > 98% |
| lineage_critical_assets_pct | % activos críticos con linaje completo | > 90% |
| pii_classification_accuracy | Precision clasificación PII | > 0.95 |
| unauthorized_access_attempts | Intentos bloqueados / semana | Tendencia decreciente |
| policy_drift_detected | Cambios no autorizados detectados | 0 críticos / trimestre |
| cost_governance_ratio | Costo gobernanza / costo total datos | < 8% |
| bias_alerts_resolved_time_min | MTTR alertas de sesgo | < 120 min |

### Flujo de Política como Código (Rego Simplificado)
1. Definir regla (ej. restricción acceso a columnas sensibles).
2. Versionar en Git y ejecutar tests (unit / integración).
3. Pipeline CI valida sintaxis + cobertura de escenarios.
4. Deploy a motor OPA/Ranger y registrar versión en catálogo.
5. Observabilidad: emitir evento OpenLineage con faceta "policyVersion".

### Ejemplo Rego (Acceso restringido a columna PII)
```rego
package access.column

default allow = false

# Entrada esperada: {"userRoles": ["analyst"], "columnsRequested": ["email","total_amount"], "piiColumns": ["email"]}
allow {
  not contains(input.columnsRequested, sensitive)
}

sensitive = col {
  col := input.piiColumns[_]
}

contains(arr, item) {
  arr[_] == item
}
```

### Ejemplo Python: Evaluación de Metadatos y Construcción de Reporte de Cumplimiento
```python
import json, datetime

ASSET = {
    "name": "ventas_limpias",
    "owner": "data-product-ventas",
    "tags": ["fact","ventas","pii:masked"],
    "pii_fields": ["email"],
    "masking_applied": True,
    "lineage_complete": True,
    "last_policy_version": "policy-gov-v23",
    "quality_checks_passed": 47,
    "quality_checks_total": 47,
    "cost_month_usd": 82.5,
    "bias_model": None
}

report = {
    "asset": ASSET["name"],
    "owner_present": ASSET.get("owner") is not None,
    "pii_protection_ok": ASSET["masking_applied"] and len(ASSET["pii_fields"])>0,
    "quality_pass_rate": ASSET["quality_checks_passed"] / ASSET["quality_checks_total"],
    "lineage_complete": ASSET["lineage_complete"],
    "policy_version": ASSET["last_policy_version"],
    "timestamp_utc": datetime.datetime.utcnow().isoformat()+"Z"
}
print(json.dumps(report, indent=2, ensure_ascii=False))
```

### Data Product Governance Blueprint (Resumen)
```yaml
data_product: ventas
version: 2025-11-23
owner: squad-ventas
slas:
  freshness_minutes: 30
  availability_pct: 99.5
classification:
  pii: [email]
  confidentiality: interno
policies:
  access: policy-gov-v23
quality_rules:
  - rule: null_rate < 0.02
  - rule: duplicate_rate < 0.01
lineage:
  tool: OpenLineage
  coverage_target_pct: 0.9
cost:
  monthly_budget_usd: 120
  alert_threshold_pct: 0.9
risk_register:
  - id: R1
    desc: Cambio esquema upstream sin aviso
    mitigation: contract + test CI
  - id: R2
    desc: Incremento PII no etiquetado
    mitigation: clasificador semanal
```

### Mapa de Riesgos 2025
| Riesgo | Impacto | Probabilidad | Mitigación |
|--------|---------|--------------|------------|
| Drift Político (policy_drift) | Alto | Bajo | Firma + revisión automatizada diff |
| Falta Owner | Medio | Medio | Workflow creación + alerta diaria |
| PII sin máscara | Alto | Bajo | Corrección automática + bloqueo acceso |
| Sesgo Modelo | Alto | Medio | Métricas fairness + retrain gatillado |
| Escalada Costos Gobernanza | Medio | Medio | FinOps revisiones mensuales |
| Brecha Linaje Crítico | Alto | Bajo | Instrumentar jobs faltantes + SLA |

### Checklist Implementación
- [ ] Repositorio policies con tests unitarios y pipeline CI.
- [ ] Instrumentación OpenLineage con faceta policyVersion.
- [ ] Clasificación PII automática (job semanal) + verificación manual.
- [ ] Owners y dominios definidos >95% activos críticos.
- [ ] Dashboard integrado: KPIs, riesgos, costos, drift, sesgo.
- [ ] Registro versionado de contratos de datos y políticas.

### Modelo de Madurez Gobernanza
| Nivel | Rasgos | Próximo Paso |
|-------|--------|-------------|
| 1 Básico | Inventario parcial | Catalogar y asignar owners |
| 2 Inicial | Políticas manuales | Introducir policy-as-code tests |
| 3 Intermedio | Linaje crítico + DQ | Añadir clasificación automática |
| 4 Avanzado | Métricas riesgo + FinOps | Integrar fairness y sesgo IA |
| 5 Optimizado | Gobernanza predictiva | Modelos de predicción incidentes |

### Referencias Verificadas
- Open Policy Agent (https://www.openpolicyagent.org)
- Apache Ranger (https://ranger.apache.org)
- DataHub (https://datahubproject.io)
- OpenLineage (https://openlineage.io)
- Immuta (https://www.immuta.com) / Privacera (https://privacera.com)
- AWS Lake Formation (https://aws.amazon.com/lake-formation/)
- Azure Purview / Microsoft Purview (https://azure.microsoft.com/services/purview/)
- GCP Data Catalog (https://cloud.google.com/data-catalog)
- Great Expectations (https://greatexpectations.io)
- dbt (https://www.getdbt.com)
- Evidently AI (https://www.evidentlyai.com)
- Fairlearn (https://fairlearn.org)
- HashiCorp Vault (https://www.vaultproject.io)

> Revisar trimestralmente: nuevas normativas (AI Act detalles finales), deprecación herramientas, ajuste metas KPI.


## Glosario KPIs Unificado 2025

Este glosario estandariza nombres, fórmula base, objetivo y propósito para los KPIs definidos en el catálogo (`04-recursos/kpi_catalog.yml`). Prefijos: dq_, lin_, gov_, fin_, ml_, arch_, sec_, val_.

| KPI | Categoría | Fórmula Resumida | Unidad | Objetivo | Propósito |
|-----|-----------|------------------|--------|----------|-----------|
| dq_null_rate | Data Quality | null_values / total_values | fracción | < 0.02 | Minimizar ausencia de datos críticos |
| dq_duplicate_rate | Data Quality | duplicate_rows / total_rows | fracción | < 0.01 | Evitar inflar métricas y sesgos |
| dq_schema_drift_events | Data Quality | breaking_changes / trimestre | eventos | 0 | Preservar contratos y compatibilidad |
| dq_materialization_staleness_pct | Data Quality | stale_materializations / total | fracción | < 0.05 | Garantizar frescura de vistas |
| lin_coverage_pct | Linaje | assets_con_linaje / total_assets | fracción | > 0.85 | Visibilidad general de flujo |
| lin_critical_assets_coverage_pct | Linaje | activos_críticos_con_linaje / activos_críticos | fracción | > 0.90 | Asegurar trazabilidad en Tier 1 |
| lin_orphan_assets_count | Linaje | activos_sin_owner_o_linaje | conteo | 0 | Reducir riesgo y deuda |
| gov_coverage_owner_pct | Gobernanza | assets_con_owner / total_assets | fracción | > 0.95 | Accountability claro |
| gov_policy_test_pass_rate | Gobernanza | tests_politica_ok / tests_totales | fracción | > 0.98 | Robustez policy-as-code |
| gov_pii_classification_accuracy | Gobernanza | TP_PII / (TP_PII + FP_PII) | fracción | > 0.95 | Confiabilidad etiquetado sensible |
| fin_waste_pct | FinOps | idle_cluster_hours / total_cluster_hours | fracción | < 0.10 | Eficiencia infraestructura |
| fin_cost_governance_ratio | FinOps | cost_gov / total_platform_cost | fracción | < 0.08 | Control costo gobernanza |
| fin_cost_per_critical_query | FinOps | costo_total_consultas_críticas / num_consultas | USD/query | -20% YoY | Optimizar eficiencia analítica |
| val_reuse_assets_pct | Valor | activos_reusados_>=2_dom / total_assets | fracción | > 0.35 | Impulsar reutilización transversal |
| arch_ttm_feature_days | Arquitectura | días(idea→uso_prod) | días | < 14 | Acelerar innovación |
| arch_storage_dup_ratio | Arquitectura | gb_duplicados / gb_total | fracción | < 0.08 | Reducir costo y complejidad |
| sec_unauthorized_access_attempts | Seguridad | intentos_bloqueados / semana | conteo | decreciente | Medir presión amenazas |
| ml_prediction_drift_psi | ML/IA | PSI(pred_actual, pred_ref) | psi | < 0.2 | Detectar degradación modelo |
| ml_drift_mttr_minutes | ML/IA | minutos(drift_detectado→mitigado) | minutos | < 180 | Velocidad recuperación |
| ml_drift_cases_resolved_pct | ML/IA | casos_drift_resueltos / total_casos | fracción | > 0.95 | Eficacia respuesta |
| gov_incidentes_regulatorios | Gobernanza | incidentes_confirmados / trimestre | conteo | 0 | Mantener cumplimiento |
| ml_roi_experimentos | ML/IA | (beneficio - costo)/costo | ratio | > 1.5 | Justificar inversión IA |

Notas:
- Freshness Tiers definidos en catálogo para interpretación consistente.
- Targets YoY requieren baseline inicial documentada.
- PSI y drift KPIs demandan referencia estable, actualizada tras retrain mayor.
- Ratio de gobernanza incluye herramienta + personal + infraestructura asociada.

> Usar este glosario como fuente única para auditorías y dashboards; cambios requieren PR con revisión de governance-office y finops-team.


### Referencias Anexos (Recursos Reutilizables 2025)

Uso rápido sin duplicar contenido:
- Contrato Datos: `04-recursos/anexos/data_contract_template.yaml`
- ADR Template: `04-recursos/anexos/adr_template.md`
- Política Column Access (Rego): `04-recursos/anexos/column_access.rego`
- Emisión Linaje: `04-recursos/helpers/event_emitter.py`
- Calidad & Drift: `04-recursos/helpers/quality_rules.py`
- Métricas FinOps: `04-recursos/helpers/cost_metrics.py`
- Catálogo KPIs: `04-recursos/kpi_catalog.yml`

> Mantener referencias vía rutas relativas; cambios requieren PR y revisión de governance-office.
